# Data Cleaning and Preprocessing: Titanic Dataset

**Oasis Infobyte Data Analytics Internship — Level 1, Task 3**

This notebook cleans the Titanic passenger dataset into an analysis-ready CSV and documents every decision.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_PATH = Path('titanic_raw.csv')
CLEAN_PATH = Path('titanic_cleaned.csv')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_columns', None)

## 1. Load data and data-quality report

In [ ]:
raw = pd.read_csv(RAW_PATH)
print('Raw shape:', raw.shape)
display(raw.head())
display(raw.dtypes.to_frame('dtype'))

quality_report = pd.DataFrame({
    'null_count': raw.isna().sum(),
    'null_percent': (raw.isna().mean() * 100).round(2),
    'unique_values': raw.nunique(dropna=True),
    'dtype': raw.dtypes.astype(str)
})
display(quality_report)
print('Exact duplicate rows:', raw.duplicated().sum())

### Initial findings

- `Age` has 177 missing values and `Embarked` has 2.
- `Cabin` is 77% missing, making record-level imputation unreliable.
- There are no exact duplicate rows.
- `PassengerId` is an identifier, while `Survived`, `Pclass`, `Sex`, and `Embarked` are categorical fields.
- Fare values include expensive first-class tickets. They require review, not automatic deletion.

## 2. Cleaning strategy and rationale

| Field / issue | Cleaning decision | Rationale |
|---|---|---|
| Column names | Convert to lowercase snake_case | Makes analysis code consistent and readable. |
| `cabin` | Drop column | 77% missing; filling cabin labels would fabricate data. |
| `age` | Median within passenger class, then overall median fallback | Class is related to travel demographics; median is robust to extreme values. |
| `embarked` | Mode imputation | Only two values are missing and the field is categorical. |
| Duplicates | Check and retain result | No exact duplicates exist, so no rows are removed. |
| Fare outliers | Retain after IQR review | High fares are plausible historical first-class fares, not data-entry errors. |

## 3. Standardisation, missing values, and data types

In [ ]:
df = raw.copy()
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))

# Standardise categorical string values before imputation.
df['sex'] = df['sex'].astype('string').str.strip().str.title()
df['embarked'] = df['embarked'].astype('string').str.strip().str.upper()

# Class-specific median age imputation, followed by an overall fallback.
df['age'] = df['age'].fillna(df.groupby('pclass')['age'].transform('median'))
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Cabin is too sparse to infer reliably.
df = df.drop(columns='cabin')

# Correct dtypes. IDs and ticket numbers remain labels, not numerical measures.
df['passengerid'] = df['passengerid'].astype('string')
df['ticket'] = df['ticket'].astype('string').str.strip()
for col in ['survived', 'pclass', 'sex', 'embarked']:
    df[col] = df[col].astype('category')
df['age'] = df['age'].astype('float64')
df['fare'] = df['fare'].astype('float64')

display(df.head())
display(df.dtypes.to_frame('clean_dtype'))
display(df.isna().sum().to_frame('remaining_nulls'))

**Decision.** The sparse `cabin` column is removed rather than guessed. Age and embarkation are imputed because their missingness is limited and the strategies can be clearly justified.

## 4. Duplicate review and outlier detection

In [ ]:
duplicates_before = int(raw.duplicated().sum())
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print(f'Exact duplicates in raw data: {duplicates_before}')
print(f'Duplicates removed after cleaning: {duplicates_removed}')

def iqr_summary(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return pd.Series({'lower_bound': lower, 'upper_bound': upper, 'outlier_count': ((series < lower) | (series > upper)).sum()})

outliers = pd.DataFrame({col: iqr_summary(df[col]) for col in ['age', 'sibsp', 'parch', 'fare']}).T
display(outliers.round(2))

fig, ax = plt.subplots(figsize=(9, 4))
df[['age', 'sibsp', 'parch', 'fare']].plot(kind='box', ax=ax)
ax.set_title('IQR Review of Numeric Fields')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'numeric_outlier_review.png', bbox_inches='tight')
plt.show()

**Decision.** IQR identifies statistically unusual fares and family counts. They are retained because they are credible Titanic passenger records. Cleaning should remove errors, not unusual but valid observations.

## 5. Before-versus-after summary and output

In [ ]:
expected_types = {
    'passengerid': 'string', 'survived': 'category', 'pclass': 'category', 'name': 'object',
    'sex': 'category', 'age': 'float', 'sibsp': 'int', 'parch': 'int',
    'ticket': 'string', 'fare': 'float', 'embarked': 'category'
}
def dtype_accuracy(frame):
    correct = sum(target in str(frame[col].dtype) for col, target in expected_types.items() if col in frame.columns)
    return f'{correct}/{len(expected_types)} expected types'

summary = pd.DataFrame({
    'metric': ['row_count', 'column_count', 'null_count', 'duplicate_count', 'dtype_accuracy'],
    'before_cleaning': [len(raw), raw.shape[1], int(raw.isna().sum().sum()), int(raw.duplicated().sum()), dtype_accuracy(raw.rename(columns=str.lower))],
    'after_cleaning': [len(df), df.shape[1], int(df.isna().sum().sum()), int(df.duplicated().sum()), dtype_accuracy(df)]
})
display(summary)

df.to_csv(CLEAN_PATH, index=False)
print(f'Cleaned dataset saved as: {CLEAN_PATH.resolve()}')

## Conclusion

The Titanic dataset is now analysis-ready: it has no missing values, no duplicate records, consistent column naming, and appropriate data types. The cleaning process preserves valid historical variation, including high fares, while avoiding unsupported guesses for the sparse cabin field.

### Key lessons

1. Missing values should be handled according to the meaning and completeness of each field.
2. Outlier detection is a review tool—not an automatic deletion rule.
3. A documented before/after report makes cleaning reproducible and trustworthy.

## Reproducibility
Run all cells from top to bottom. The final file is saved as `titanic_cleaned.csv`.